In [ ]:
pip install torch-geometric


In [ ]:
!pip install -q rdflib

import os
import glob
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report
from rdflib import Graph, RDFS


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = ''
DATA_DIR = os.path.join(BASE_DIR, 'dados_brutos')

BLACK_DIR = os.path.join(DATA_DIR, 'black')
WHITE_DIR = os.path.join(DATA_DIR, 'white')
VAL_DIR   = os.path.join(DATA_DIR, 'validacao')

ONT_PATH  = os.path.join(BASE_DIR, 'ontologia', 'tinilau.owl')

WINDOW_SIZE = 10
BATCH_SIZE = 2048


In [ ]:
ontology = Graph()
ontology.parse(ONT_PATH)

ontology_labels = set()

for s,p,o in ontology.triples((None, RDFS.label, None)):
    ontology_labels.add(str(o).lower())

print("Conceitos ontológicos:", len(ontology_labels))


Conceitos ontológicos: 315


In [ ]:
def detect_data_sheet(path):
    xls = pd.ExcelFile(path)
    for s in xls.sheet_names:
        if "Explain" in s:
            continue
        tmp = pd.read_excel(path, sheet_name=s, nrows=3)
        if "MMSI" in tmp.columns:
            return s
    return None


def parse_coordinates(df):
    coords = df["Position Coordinates"].str.split(",", expand=True)
    df["latitude"] = coords[0].str.strip().astype(float)
    df["longitude"] = coords[1].str.strip().astype(float)
    return df


def load_folder(folder, label=None):

    files = glob.glob(os.path.join(folder, "*.xlsx"))
    dfs = []

    for f in files:
        if "-sightings" in f:
            continue

        sheet = detect_data_sheet(f)
        if sheet is None:
            continue

        df = pd.read_excel(f, sheet_name=sheet)
        df = parse_coordinates(df)

        df["timestamp"] = pd.to_datetime(df["Position Instant"], errors="coerce")
        df["mmsi"] = df["MMSI"]

        if label is not None:
            df["label"] = label

        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)


In [ ]:
df_black = load_folder(BLACK_DIR, 1)
df_white = load_folder(WHITE_DIR, 0)
df_val   = load_folder(VAL_DIR, None)

df_train_full = pd.concat([df_black, df_white], ignore_index=True)
df_train_full = df_train_full.sort_values(["mmsi", "timestamp"])
df_val = df_val.sort_values(["mmsi", "timestamp"])


In [ ]:
for df in [df_train_full, df_val]:
    df["lat_raw"] = df["latitude"]
    df["lon_raw"] = df["longitude"]
    df["speed_raw"] = df["Speed"]
    df["vessel_type_raw"] = df["Vessel Type"]


In [ ]:
def map_to_ontology(vtype):

    if pd.isna(vtype):
        return "unknown"

    vtype = str(vtype).lower()

    for label in ontology_labels:
        if label in vtype:
            return label

    return "other"

df_train_full["ontology_class"] = df_train_full["vessel_type_raw"].apply(map_to_ontology)
df_val["ontology_class"] = df_val["vessel_type_raw"].apply(map_to_ontology)


In [ ]:
def add_angular(df, col):
    rad = np.deg2rad(df[col].fillna(0))
    df[f"{col}_sin"] = np.sin(rad)
    df[f"{col}_cos"] = np.cos(rad)

for df in [df_train_full, df_val]:
    add_angular(df, "Heading")
    add_angular(df, "Course")

FEATURES = [
    "latitude", "longitude",
    "Speed", "Acceleration",
    "Heading_sin", "Heading_cos",
    "Course_sin", "Course_cos"
]

df_train_full = df_train_full.dropna(subset=FEATURES).copy()
df_val = df_val.dropna(subset=FEATURES).copy()


In [ ]:
scaler = StandardScaler()

df_train_full.loc[:, FEATURES] = scaler.fit_transform(df_train_full[FEATURES])
df_val.loc[:, FEATURES] = scaler.transform(df_val[FEATURES])


In [ ]:
def create_windows(df, has_label=True):

    X, y, coords, speeds, types, vessels = [], [], [], [], [], []

    for mmsi, group in df.groupby("mmsi"):

        group = group.sort_values("timestamp")

        values = group[FEATURES].values
        lat_raw = group["lat_raw"].values
        lon_raw = group["lon_raw"].values
        speed_raw = group["speed_raw"].values
        ont_class = group["ontology_class"].iloc[0]

        if has_label:
            labels = group["label"].values

        for i in range(len(group) - WINDOW_SIZE):

            X.append(values[i:i+WINDOW_SIZE])
            vessels.append(mmsi)

            coords.append(
                np.stack([
                    lat_raw[i:i+WINDOW_SIZE],
                    lon_raw[i:i+WINDOW_SIZE]
                ], axis=1)
            )

            speeds.append(speed_raw[i:i+WINDOW_SIZE])
            types.append(ont_class)

            if has_label:
                y.append(labels[i+WINDOW_SIZE-1])

    if has_label:
        return np.array(X), np.array(y), np.array(coords), np.array(speeds), np.array(types), np.array(vessels)
    else:
        return np.array(X), np.array(coords), np.array(speeds), np.array(types), np.array(vessels)


In [ ]:
X, y, coords, speeds, types, vessels = create_windows(df_train_full, True)
X_val, coords_val, speeds_val, types_val, vessels_val = create_windows(df_val, False)


In [ ]:
mmsi_labels = {}

for mmsi in np.unique(vessels):
    mask = vessels == mmsi
    mmsi_labels[mmsi] = y[mask][0]

mmsis = np.array(list(mmsi_labels.keys()))
labels_mmsi = np.array([mmsi_labels[m] for m in mmsis])

from sklearn.model_selection import train_test_split

train_mmsi, test_mmsi = train_test_split(
    mmsis,
    test_size=0.2,
    stratify=labels_mmsi,
    random_state=42
)

train_mask = np.isin(vessels, train_mmsi)
test_mask  = np.isin(vessels, test_mmsi)

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

coords_train, coords_test = coords[train_mask], coords[test_mask]
speeds_train, speeds_test = speeds[train_mask], speeds[test_mask]

types_train, types_test = types[train_mask], types[test_mask]


type_ids_train = np.array([type_map[t] for t in types_train])
type_ids_test  = np.array([type_map[t] for t in types_test])

coords_mean_np = coords_train.reshape(-1, 2).mean(axis=0)
coords_std_np  = coords_train.reshape(-1, 2).std(axis=0)

coords_train = (coords_train - coords_mean_np) / coords_std_np
coords_test  = (coords_test  - coords_mean_np) / coords_std_np

coords_val = (coords_val - coords_mean_np) / coords_std_np

coords_mean = torch.tensor(coords_mean_np, dtype=torch.float32).to(device)
coords_std  = torch.tensor(coords_std_np, dtype=torch.float32).to(device)

print("Distribuição y_train:", np.unique(y_train, return_counts=True))
print("Distribuição y_test :", np.unique(y_test, return_counts=True))


In [ ]:
all_types = np.unique(np.concatenate([types, types_val]))
type_map = {t:i for i,t in enumerate(all_types)}

type_ids_train = np.array([type_map[t] for t in types_train])
type_ids_test  = np.array([type_map[t] for t in types_test])
type_ids_val   = np.array([type_map[t] for t in types_val])


In [ ]:
def haversine_torch(lat1, lon1, lat2, lon2):

    R = 6371000

    lat1 = torch.deg2rad(lat1)
    lon1 = torch.deg2rad(lon1)
    lat2 = torch.deg2rad(lat2)
    lon2 = torch.deg2rad(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = torch.sin(dlat/2)**2 + torch.cos(lat1)*torch.cos(lat2)*torch.sin(dlon/2)**2
    c = 2 * torch.atan2(torch.sqrt(a), torch.sqrt(1-a))

    return R * c


In [ ]:
class OPCTM(nn.Module):

    def __init__(self, input_dim, num_types):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, 128)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=4,
            dropout=0.2,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.state_predictor = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 2)   # agora só latitude e longitude
        )


        self.type_embedding = nn.Embedding(num_types, 64)
        self.ontology_proj = nn.Linear(64, 128)

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x, type_ids):

        x = self.input_proj(x)
        z = self.transformer(x)
        z = z.mean(dim=1)

        pred_state = self.state_predictor(z)

        ont = self.type_embedding(type_ids)
        ont = self.ontology_proj(ont)

        logits = self.classifier(z)

        return z, pred_state, ont, logits


In [ ]:
def evaluate_model(model, X_test, y_test, type_ids_test):

    model.eval()

    all_probs = []
    all_true = []

    with torch.no_grad():

        for i in range(0, len(X_test), BATCH_SIZE):

            xb = torch.tensor(X_test[i:i+BATCH_SIZE], dtype=torch.float32).to(device)
            yb = torch.tensor(y_test[i:i+BATCH_SIZE], dtype=torch.long).to(device)
            tb = torch.tensor(type_ids_test[i:i+BATCH_SIZE], dtype=torch.long).to(device)

            _, _, _, logits = model(xb, tb)

            probs = torch.softmax(logits, dim=1)[:,1]

            all_probs.extend(probs.cpu().numpy())
            all_true.extend(yb.cpu().numpy())

    return roc_auc_score(all_true, all_probs)


In [ ]:
def physics_error(coords, speeds, per_sample=False):
    """
    coords: (B, T, 2) latitude/longitude em graus (RAW)
    speeds: (B, T) velocidade em nós (RAW)
    """

    lat = coords[:, :, 0]
    lon = coords[:, :, 1]

    dist = haversine_torch(
        lat[:, :-1], lon[:, :-1],
        lat[:, 1:],  lon[:, 1:]
    ) / 1000.0

    speed_kmh = speeds[:, :-1] * 1.852

    expected = speed_kmh / 60.0

    error = dist - expected
    sq_error = error ** 2

    if per_sample:
        # Retorna erro médio por amostra (B,)
        return sq_error.mean(dim=1)
    else:
        # Retorna escalar médio global
        return sq_error.mean()


In [ ]:
model = OPCTM(len(FEATURES), len(all_types)).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

λ1, λ2, λ3, λ4 = 1.0, 1.0, 0.5, 0.3

for epoch in range(20):

    model.train()

    total_loss = 0.0
    total_samples = 0

    sum_L_pred = 0.0
    sum_L_phys = 0.0
    sum_L_ont  = 0.0
    sum_L_cls  = 0.0

    perm = np.random.permutation(len(X_train))

    for i in range(0, len(X_train), BATCH_SIZE):

        batch_idx = perm[i:i+BATCH_SIZE]

        xb = torch.tensor(X_train[batch_idx], dtype=torch.float32).to(device)
        yb = torch.tensor(y_train[batch_idx], dtype=torch.long).to(device)
        cb = torch.tensor(coords_train[batch_idx], dtype=torch.float32).to(device)
        sb = torch.tensor(speeds_train[batch_idx], dtype=torch.float32).to(device)
        tb = torch.tensor(type_ids_train[batch_idx], dtype=torch.long).to(device)

        optimizer.zero_grad()

        z, pred_state, ont, logits = model(xb, tb)

        # =====================================================
        # =====================================================
        true_next = cb[:, -1, :]  # já normalizado
        L_pred = F.mse_loss(pred_state, true_next, reduction='mean')

        # =====================================================
        # =====================================================

        lat_prev = cb[:, -2, 0] * coords_std[0] + coords_mean[0]
        lon_prev = cb[:, -2, 1] * coords_std[1] + coords_mean[1]

        lat_pred = pred_state[:, 0] * coords_std[0] + coords_mean[0]
        lon_pred = pred_state[:, 1] * coords_std[1] + coords_mean[1]

        dist_pred = haversine_torch(
            lat_prev, lon_prev,
            lat_pred, lon_pred
        ) / 1000.0

        speed_kmh = sb[:, -2] * 1.852
        expected = speed_kmh / 60.0

        L_phys = torch.mean(torch.log1p((dist_pred - expected) ** 2))

        # =====================================================
        # =====================================================
        L_ont = F.mse_loss(z, ont, reduction='mean')

        # =====================================================
        # =====================================================
        L_cls = F.cross_entropy(logits, yb)

        # =====================================================
        # LOSS FINAL
        # =====================================================
        loss = λ1*L_pred + λ2*L_phys + λ3*L_ont + λ4*L_cls

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        n = xb.size(0)

        total_loss += loss.item() * n
        total_samples += n

        sum_L_pred += L_pred.item() * n
        sum_L_phys += L_phys.item() * n
        sum_L_ont  += L_ont.item()  * n
        sum_L_cls  += L_cls.item()  * n

    epoch_loss = total_loss / total_samples

    print(f"\nEpoch {epoch}")
    print(f"Loss médio: {epoch_loss:.6f}")
    print(f"L_pred: {sum_L_pred/total_samples:.6f}")
    print(f"L_phys: {sum_L_phys/total_samples:.6f}")
    print(f"L_ont : {sum_L_ont/total_samples:.6f}")
    print(f"L_cls : {sum_L_cls/total_samples:.6f}")


In [ ]:
model.eval()

all_probs = []
all_preds = []
all_true  = []

with torch.no_grad():

    for i in range(0, len(X_test), BATCH_SIZE):

        xb = torch.tensor(X_test[i:i+BATCH_SIZE], dtype=torch.float32).to(device)
        yb = torch.tensor(y_test[i:i+BATCH_SIZE], dtype=torch.long).to(device)
        tb = torch.tensor(type_ids_test[i:i+BATCH_SIZE], dtype=torch.long).to(device)

        z, pred_state, ont, logits = model(xb, tb)

        probs = torch.softmax(logits, dim=1)[:,1]

        preds = torch.argmax(logits, dim=1)

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(yb.cpu().numpy())

all_probs = np.array(all_probs)
all_preds = np.array(all_preds)
all_true  = np.array(all_true)

print("AUC:", roc_auc_score(all_true, all_probs))
print(classification_report(all_true, all_preds))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

sns.set_style("whitegrid")
plt.rcParams.update({'font.size': 12})

# =========================================================
# =========================================================

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
cm = confusion_matrix(all_true, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['White (Normal)', 'Black (Spoof)'],
            yticklabels=['White (Normal)', 'Black (Spoof)'])
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title('Matriz de Confusão')

# --- Curva ROC ---
plt.subplot(1, 2, 2)
fpr, tpr, _ = roc_curve(all_true, all_probs)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Curva ROC (area = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taxa de Falsos Positivos')
plt.ylabel('Taxa de Verdadeiros Positivos')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# =========================================================
def haversine_safe(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1, lat2, lon2 = map(torch.deg2rad, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = torch.sin(dlat/2)**2 + torch.cos(lat1)*torch.cos(lat2)*torch.sin(dlon/2)**2

    a = torch.clamp(a, min=0.0, max=1.0)

    c = 2 * torch.atan2(torch.sqrt(a), torch.sqrt(1-a))
    return R * c

# =========================================================

model.eval()
risk_scores = []

coords_mean_t = coords_mean.to(device)
coords_std_t  = coords_std.to(device)

with torch.no_grad():
    for i in range(0, len(X_val), BATCH_SIZE):
        xb = torch.tensor(X_val[i:i+BATCH_SIZE], dtype=torch.float32).to(device)
        cb = torch.tensor(coords_val[i:i+BATCH_SIZE], dtype=torch.float32).to(device)
        sb = torch.tensor(speeds_val[i:i+BATCH_SIZE], dtype=torch.float32).to(device)
        tb = torch.tensor(type_ids_val[i:i+BATCH_SIZE], dtype=torch.long).to(device)

        z, pred_state, ont, logits = model(xb, tb)

        true_next = cb[:, -1, :]
        L_pred = F.mse_loss(pred_state, true_next, reduction='none').mean(dim=1)

        lat_prev = cb[:, -2, 0] * coords_std_t[0] + coords_mean_t[0]
        lon_prev = cb[:, -2, 1] * coords_std_t[1] + coords_mean_t[1]
        lat_pred = pred_state[:, 0] * coords_std_t[0] + coords_mean_t[0]
        lon_pred = pred_state[:, 1] * coords_std_t[1] + coords_mean_t[1]

        dist_pred = haversine_safe(lat_prev, lon_prev, lat_pred, lon_pred) / 1000.0 # km

        speed_kmh = sb[:, -2] * 1.852
        expected = speed_kmh / 60.0 # km/min

        diff = (dist_pred - expected) ** 2
        diff = torch.clamp(diff, max=1e15) # Evita explosão numérica
        L_phys = torch.log1p(diff)

        L_ont = F.mse_loss(z, ont, reduction='none').mean(dim=1)

        risk = L_pred + 0.3*L_phys + 0.5*L_ont

        risk_scores.extend(risk.cpu().numpy())

risk_scores = np.array(risk_scores)

risk_scores = np.nan_to_num(risk_scores, nan=0.0, posinf=risk_scores[~np.isinf(risk_scores)].max())

print(f"Pronto! Min: {risk_scores.min():.4f}, Max: {risk_scores.max():.4f}, Média: {risk_scores.mean():.4f}")

# =========================================================
plt.figure(figsize=(10, 6))

limite_visual = np.percentile(risk_scores, 99)
dados_para_plotar = risk_scores[risk_scores < limite_visual]

sns.histplot(dados_para_plotar, bins=50, kde=True, color='purple', alpha=0.6)

plt.axvline(risk_scores.mean(), color='red', linestyle='--', label=f'Média Global')
plt.axvline(risk_scores.mean() + 2*risk_scores.std(), color='orange', linestyle='--', label='Limiar (Média + 2std)')

plt.title(f'Distribuição de Risco (Validation Set)\n(Visualizando 99% dos dados - corte em {limite_visual:.2f})')
plt.xlabel('Score de Risco')
plt.ylabel('Frequência')
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

# =========================================================
precision, recall, thresholds = precision_recall_curve(all_true, all_probs)
avg_precision = average_precision_score(all_true, all_probs)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='purple', lw=2, label=f'PR Curve (AP = {avg_precision:.3f})')
plt.xlabel('Recall (Capacidade de encontrar Spoofers)')
plt.ylabel('Precision (Certeza quando diz que é Spoofer)')
plt.title('Curva Precision-Recall')
plt.legend(loc="lower left")
plt.grid(True)
plt.show()
